# 008 — Thresholds for the XGBoost model

Step 1.7 made XGBoost the product model. Its risk score is the **predicted fraud probability**, so the Isolation Forest thresholds from Step 1.6 (0.983 on a percentile scale) no longer mean anything.

This redoes Step 1.6 for XGBoost with the same cost model. One difference: XGBoost's scores on its own training rows are overconfident, so thresholds are chosen on **out-of-fold** probabilities. The training data is cut into 5 folds and each fold is scored by a model that never saw it. The test set is used once, at the end.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import settings
from src.ml.evaluate import summary_metrics, threshold_table
from src.ml.isolation_forest import fit_risk_model
from src.ml.model import fit_xgboost, fraud_probability, out_of_fold_probability
from src.ml.preprocess import load_splits
from src.ml.thresholds import (
    PROBABILITY_THRESHOLDS, Costs, approve_all_cost_per_100k, choose_thresholds,
    costs_from_settings, policy_costs,
)

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.width", 200)

## 2. Out-of-fold probabilities and the final model

In [2]:
X_train, X_test, y_train, y_test = load_splits()
amount_train, amount_test = X_train["Amount"], X_test["Amount"]

start = time.perf_counter()
oof_proba = out_of_fold_probability(X_train, y_train, n_splits=5)
print(f"out-of-fold probabilities for {len(oof_proba):,} training rows in {time.perf_counter() - start:.0f}s")

start = time.perf_counter()
model = fit_xgboost(X_train, y_train)
print(f"final model on the training data in {time.perf_counter() - start:.0f}s")

refit_proba = fraud_probability(model, X_train)
test_proba = fraud_probability(model, X_test)

out-of-fold probabilities for 226,980 training rows in 35s


final model on the training data in 4s


## 3. Why out-of-fold scores are needed

In [3]:
pd.DataFrame(
    {
        "final model re-scoring training rows": summary_metrics(y_train, refit_proba),
        "out-of-fold, training rows": summary_metrics(y_train, oof_proba),
        "final model, test set": summary_metrics(y_test, test_proba),
    }
).T[["n_fraud", "pr_auc", "roc_auc"]]

,n_fraud,pr_auc,roc_auc
final model re-scoring training rows,378.0000,0.9333,0.9938
"out-of-fold, training rows",378.0000,0.8478,0.9743
"final model, test set",95.0000,0.8207,0.9696


## 4. Price every threshold pair and choose

In [4]:
costs = costs_from_settings()
oof_costs = policy_costs(
    y_train, oof_proba, amount_train, costs,
    review_candidates=PROBABILITY_THRESHOLDS,
    block_candidates=(*PROBABILITY_THRESHOLDS, None),
)
choice = choose_thresholds(oof_costs, max_review_rate=settings.max_review_rate)
approve_all = approve_all_cost_per_100k(y_train, amount_train, costs)

print(f"priced {len(oof_costs):,} (review, block) pairs on out-of-fold probabilities")
print(f"review threshold: {choice.review_threshold}")
print(f"block threshold:  {choice.block_threshold}")
print(f"reviewed {choice.review_rate:.3%} | blocked {choice.block_rate:.3%}")
print(f"cost ${choice.cost_per_100k:,.0f} per 100k vs ${approve_all:,.0f} with no model "
      f"({1 - choice.cost_per_100k / approve_all:.0%} saved)")

oof_costs.nsmallest(5, "cost_per_100k")[
    ["review_threshold", "block_threshold", "review_rate", "block_rate", "fraud_stopped", "cost_per_100k"]
]

priced 5,994 (review, block) pairs on out-of-fold probabilities
review threshold: 0.24
block threshold:  0.95
reviewed 0.050% | blocked 0.099%
cost $5,453 per 100k vs $21,806 with no model (75% saved)


,review_threshold,block_threshold,review_rate,block_rate,fraud_stopped,cost_per_100k
2413,0.1600,0.8900,0.0004,0.0012,0.8307,"5,369.2087"
2497,0.1700,0.8900,0.0004,0.0012,0.8307,"5,369.2087"
2580,0.1800,0.8900,0.0004,0.0012,0.8280,"5,369.5083"
2743,0.2000,0.8900,0.0003,0.0012,0.8254,"5,369.9489"
2823,0.2100,0.8900,0.0003,0.0012,0.8228,"5,370.3895"


## 5. Precision at the chosen thresholds (out-of-fold)

Blocking only pays above 90% precision (Step 1.6: 1 - $5 review / $50 false block).

In [5]:
chosen = [choice.review_threshold] + ([choice.block_threshold] if choice.block_threshold is not None else [])
threshold_table(y_train, oof_proba, thresholds=chosen, amounts=amount_train)

,threshold,flagged,flagged_%,tp,fp,fn,precision,recall,f1,normal_flagged_%,alerts_per_fraud,fraud_amount_caught_%
0,0.2400,338,0.1489,309,29,69,0.9142,0.8175,0.8631,0.0128,1.0939,75.9793
1,0.9500,225,0.0991,220,5,158,0.9778,0.5820,0.7297,0.0022,1.0227,54.9966


## 6. How much do the assumptions matter?

In [6]:
rows = []
for review_cost in [2, 5, 10]:
    for false_block_cost in [20, 50, 200]:
        scenario = Costs(review_cost=review_cost, false_block_cost=false_block_cost,
                         chargeback_fee=costs.chargeback_fee)
        table = policy_costs(y_train, oof_proba, amount_train, scenario,
                             review_candidates=PROBABILITY_THRESHOLDS,
                             block_candidates=(*PROBABILITY_THRESHOLDS, None))
        pick = choose_thresholds(table, max_review_rate=settings.max_review_rate)
        baseline = approve_all_cost_per_100k(y_train, amount_train, scenario)
        rows.append({
            "review_cost": review_cost,
            "false_block_cost": false_block_cost,
            "review_threshold": pick.review_threshold,
            "block_threshold": pick.block_threshold,
            "reviewed_%": pick.review_rate * 100,
            "blocked_%": pick.block_rate * 100,
            "saving_vs_no_model_%": 100 * (1 - pick.cost_per_100k / baseline),
        })
pd.DataFrame(rows)

,review_cost,false_block_cost,review_threshold,block_threshold,reviewed_%,blocked_%,saving_vs_no_model_%
0,2,20,0.0040,0.9800,0.5080,0.0590,78.9669
1,2,50,0.0040,0.9900,0.5282,0.0388,78.8618
2,2,200,0.0040,NaN,0.5670,0.0000,78.5062
3,5,20,0.2100,0.9500,0.0515,0.0991,75.3416
4,5,50,0.2400,0.9500,0.0498,0.0991,74.9933
5,5,200,0.2400,0.9900,0.1101,0.0388,74.1144
6,10,20,0.2400,0.8900,0.0308,0.1181,74.9832
7,10,50,0.2400,0.9300,0.0383,0.1106,74.2760
8,10,200,0.2400,0.9500,0.0498,0.0991,72.3364


## 7. Confirm once on the test set

In [7]:
def price(y, scores, amounts, review, block):
    row = policy_costs(y, scores, amounts, costs, review_candidates=[review], block_candidates=[block]).iloc[0]
    baseline = approve_all_cost_per_100k(y, amounts, costs)
    return {
        "reviewed_%": row["review_rate"] * 100,
        "blocked_%": row["block_rate"] * 100,
        "fraud_stopped_%": row["fraud_stopped"] * 100,
        "fraud_loss_stopped_%": row["fraud_loss_stopped"] * 100,
        "cost_per_100k": row["cost_per_100k"],
        "no_model_cost_per_100k": baseline,
        "saving_%": 100 * (1 - row["cost_per_100k"] / baseline),
    }

pd.DataFrame({
    "train (out-of-fold)": price(y_train, oof_proba, amount_train, choice.review_threshold, choice.block_threshold),
    "test": price(y_test, test_proba, amount_test, choice.review_threshold, choice.block_threshold),
})

,train (out-of-fold),test
reviewed_%,0.0498,0.0828
blocked_%,0.0991,0.0564
fraud_stopped_%,81.7460,75.7895
fraud_loss_stopped_%,76.6399,74.3282
cost_per_100k,"5,452.9474","7,739.0477"
no_model_cost_per_100k,"21,805.9212","28,532.9539"
saving_%,74.9933,72.8768


In [8]:
block_at = np.inf if choice.block_threshold is None else choice.block_threshold
decision = np.select([test_proba >= block_at, test_proba >= choice.review_threshold], ["block", "review"], default="approve")
actual = np.where(y_test.to_numpy() == 1, "fraud", "normal")
pd.crosstab(pd.Series(decision, name="decision"), pd.Series(actual, name="actual")).reindex(["approve", "review", "block"])

actual,fraud,normal
decision,,
approve,23,56644
review,40,7
block,32,0


## 8. Against the Isolation Forest policy from Step 1.6 (test set)

In [9]:
if_risk = fit_risk_model(X_train, y_train).risk(X_test)
pd.DataFrame({
    "Isolation Forest (review >= 0.983, never block)": price(y_test, if_risk, amount_test, 0.983, None),
    "XGBoost (thresholds above)": price(y_test, test_proba, amount_test, choice.review_threshold, choice.block_threshold),
}).T

,reviewed_%,blocked_%,fraud_stopped_%,fraud_loss_stopped_%,cost_per_100k,no_model_cost_per_100k,saving_%
"Isolation Forest (review >= 0.983, never block)",1.8768,0.0000,66.3158,67.0581,"18,783.2094","28,532.9539",34.1701
XGBoost (thresholds above),0.0828,0.0564,75.7895,74.3282,"7,739.0477","28,532.9539",72.8768


## Findings: XGBoost thresholds

**Decision: review at fraud probability >= 0.24, block automatically at >= 0.95.**

> We block when the predicted probability is at least 0.95, because out-of-fold 98% of those transactions were fraud and a $50 false block is rare there. Transactions from 0.24 to 0.95 go to an analyst, because a $5 review is cheaper than letting that fraud through.

**Out-of-fold scores are realistic; a model re-scoring its own training rows is not**

| scores used | PR-AUC |
|---|---|
| final model re-scoring its training rows | 0.933 (overconfident) |
| out-of-fold, training rows | 0.848 |
| final model, test set | 0.821 |

Out-of-fold is close to the test number, which is why the thresholds are chosen on it.

**Chosen on out-of-fold probabilities** (378 training fraud)
- 5,994 threshold pairs priced. The cheapest was review 0.16 / block 0.89 ($5,369 per 100k). Within the 2% tie band, 0.24 / 0.95 blocks fewer customers, at $5,453.
- At >= 0.24: 91% precision, 82% recall. At >= 0.95: 98% precision, 58% recall.
- Reviews 0.05% of transactions and blocks 0.10%. That is **75% cheaper** than no model.

**Confirmed once on the test set**

| | train (out-of-fold) | test |
|---|---|---|
| reviewed | 0.050% | 0.083% |
| blocked | 0.099% | 0.056% |
| fraud cases stopped | 81.7% | 75.8% |
| fraud losses stopped | 76.6% | 74.3% |
| saving vs no model | 75% | 73% |

Test-set decisions (95 fraud, 56,651 normal):
- **Blocked 32, all fraud.** No legitimate customer was blocked.
- Reviewed 47: 40 fraud, 7 normal.
- Approved the rest: 23 fraud got through.

**Caveat: the probability scale shifts between models.** On test the final model blocks about half as often as the out-of-fold models did (0.056% vs 0.099%) and reviews more. Some fraud moves from the block tier into review, where it is still stopped, so the cost barely changes. But thresholds are tied to the model that produced them: re-check them every time the model is retrained.

**What the assumptions change**
- Review cost $5 or $10, false-block cost $20 to $200: review threshold 0.21 to 0.24, block threshold 0.89 to 0.99.
- Review cost $2: reviews become cheap enough to lower the review threshold to 0.004 (about 0.5% of traffic reviewed). With $2 reviews *and* $200 false blocks, auto-blocking turns off.
- Savings stay between 72% and 79% in every scenario.

**Against the Isolation Forest policy from Step 1.6 (test set)**

| | reviewed | blocked | fraud stopped | saving vs no model |
|---|---|---|---|---|
| Isolation Forest (review >= 0.983) | 1.88% | 0% | 66% | 34% |
| **XGBoost (review >= 0.24, block >= 0.95)** | **0.08%** | **0.06%** | **76%** | **73%** |

About 23x fewer transactions to review, 10 more points of fraud stopped, and twice the saving.

**Config updated:** `REVIEW_THRESHOLD=0.24`, `BLOCK_THRESHOLD=0.95` in `.env` and `.env.example`; `model_filename` is now `fraud_model.joblib`.